In [46]:
import pandas as pd

In [47]:

import pandas as pd

countryC = pd.read_csv(
    "https://raw.githubusercontent.com/Imagin-io/country-nationality-list/master/countries.csv",
    encoding="utf-8"
)

countryC = countryC.rename(
    columns={
        "alpha_3_code": "country_code",
        "nationality": "nationality"
    }
)

countryC = countryC[["country_code", "nationality"]].drop_duplicates()

# Add missing Formula 1 nationality codes
extra_countries = pd.DataFrame({
    "country_code": [
        "DEN", "MON", "NED", "GER", "RSA", "SUI",
        "GBR", "POR", "URS", "ROC", "BUL"
    ],
    "nationality": [
        "Danish", "Monegasque", "Dutch", "German",
        "South African", "Swiss", "British", "Portuguese",
        "Soviet", "Russian", "Bulgarian"
    ]
})

# Add only codes not already present
countryC = pd.concat(
    [
        countryC,
        extra_countries[
            ~extra_countries["country_code"].isin(countryC["country_code"])
        ]
    ],
    ignore_index=True
)

# Remove missing values and duplicates
countryC = countryC.dropna(
    subset=["country_code", "nationality"]
).drop_duplicates(subset=["country_code"])

# Save CSV with UTF-8 encoding (Excel compatible)
countryC.to_csv(
    "cleaned/Country_Codes.csv",
    index=False,
    encoding="utf-8-sig"
)


In [48]:
def endid(df):
    ids = [df.columns[x] for x in range(len(df.columns)) if df.columns[x].endswith("_id")]
    ids2 = [df.columns[x] for x in range(len(df.columns)) if df.columns[x] not in ids]

    return df[ids2 + ids]

In [49]:

import pandas as pd

# =========================
# LOAD DATA
# =========================

drivers = pd.read_csv("formula_one_driver.csv").drop(
    columns=["abbreviation", "api_id", "permanent_car_number", "reference", "wikipedia"]
).rename(columns={"id": "driver_id"})

df = pd.read_csv("formula_one_teamdriver.csv").drop(
    columns=["api_id", "role"]
)

df = df.drop(columns=["id"])

teams = pd.read_csv("formula_one_team.csv").drop(
    columns=["base_team_id", "api_id", "primary_color", "reference", "wikipedia"]
).rename(columns={"id": "team_id"})

season = pd.read_csv("formula_one_season.csv").drop(
    columns=["api_id", "wikipedia"]
).rename(columns={"id": "season_id"})


# =========================
# DRIVERS CLEANING
# =========================

drivers = drivers.rename(columns={
    col: f"drivers_{col}" for col in drivers.columns
    if "id" not in col
})

cols = [
    "drivers_forename",
    "drivers_surname",
    "drivers_date_of_birth",
    "drivers_nationality",
    "drivers_country_code",
    "driver_id",
]

drivers = drivers[cols]


# =========================
# FILL MISSING BIRTHDATES
# =========================

missingBds = {
    "864": "1999-05-06",
    "865": "2001-09-22",
    "866": "2000-10-14",
    "867": "2002-01-31",
    "868": "2004-02-04",
    "869": "2005-05-02",
    "870": "2005-03-07",
    "873": "2004-01-19",
    "874": "2005-11-11",
    "875": "2001-06-16",
    "876": "2002-01-13",
    "877": "1996-05-20",
    "878": "2000-05-23",
    "879": "2000-03-30",
    "880": "2004-12-03",
    "881": "2008-08-09",
    "882": "2006-12-21",
    "883": "2005-03-20"
}

for idx, birthdate in missingBds.items():
    drivers.loc[int(idx), "drivers_date_of_birth"] = birthdate


# =========================
# FILL MISSING NATIONALITIES
# =========================

def find_col(df, substring):
    matches = [c for c in df.columns if substring in c]

    if not matches:
        raise KeyError(
            f"Nessuna colonna contenente '{substring}' in {list(df.columns)}"
        )

    return matches[0]


drv_code_col = find_col(drivers, "country_code")
drv_nat_col = find_col(drivers, "nationality")
cnt_code_col = find_col(countryC, "country_code")
cnt_nat_col = find_col(countryC, "nationality")


# Create nationality mapping
nat_map = (
    countryC[[cnt_code_col, cnt_nat_col]]
    .dropna(subset=[cnt_code_col, cnt_nat_col])
    .drop_duplicates(subset=[cnt_code_col])
    .set_index(cnt_code_col)[cnt_nat_col]
    .to_dict()
)


# Update only drivers with missing nationality
target_ids = drivers.loc[
    drivers[drv_nat_col].isna() & drivers[drv_code_col].notna(),
    "driver_id"
]

mask = drivers["driver_id"].isin(target_ids)


drivers.loc[mask, drv_nat_col] = (
    drivers.loc[mask, drv_code_col]
    .map(nat_map)
    .fillna(drivers.loc[mask, drv_nat_col])
)


# =========================
# TEAMS CLEANING
# =========================

teams = teams.rename(columns={
    col: f"teams_{col}" for col in teams.columns
    if "id" not in col
})

teams = teams[
    [
        "teams_name",
        "teams_nationality",
        "teams_country_code",
        "team_id"
    ]
]


# =========================
# SEASONS CLEANING
# =========================

season = season.rename(columns={
    col: f"season_{col}" for col in season.columns
    if "id" not in col
})

season = season[
    [
        "season_year",
        "season_id",
        "championship_system_id"
    ]
]


# =========================
# EXPORT CLEANED DATA
# =========================

drivers.to_csv(
    "cleaned/Drivers_cleaned.csv",
    index=False,
    encoding="utf-8-sig"
)

teams.to_csv(
    "cleaned/Teams_cleaned.csv",
    index=False,
    encoding="utf-8-sig"
)

season.to_csv(
    "cleaned/Season_cleaned.csv",
    index=False,
    encoding="utf-8-sig"
)


# =========================
# MERGE DATAFRAMES
# =========================

df = pd.merge(df, drivers, on="driver_id", how="left")
df = pd.merge(df, teams, on="team_id", how="left")
df = pd.merge(df, season, on="season_id", how="left")

df = endid(df)


# =========================
# EXPORT MERGED DATA
# =========================

df.to_csv(
    "cleaned/Merged_drivers_info.csv",
    index=False,
    encoding="utf-8-sig"
)


In [50]:
def checkforNAN(df):
    missing = df.isna() | df.astype("string").apply(
        lambda col: col.str.strip().eq("")
    )

    missing_cols = missing.columns[missing.any()].tolist()

    if missing_cols:
        print("columns with missing values:")
        for col in missing_cols:
            print(f"{col}: {missing[col].sum()} missing values")
    else:
        print("No missing values.")

In [51]:
import pandas as pd

# =========================
# CIRCUITS
# =========================

circuit = pd.read_csv("formula_one_circuit.csv").drop(columns=["api_id", "reference", "wikipedia"])
circuit = circuit.rename(columns={"id": "circuit_id"})
circuit = circuit[["name", "locality", "altitude", "longitude", "country", "country_code", "circuit_id"]]
circuit = circuit.rename(columns={col: f"circuit_{col}" for col in circuit.columns if "id" not in col})

# =========================
# ROUNDS
# =========================

round = pd.read_csv("formula_one_round.csv")
round = round[round["is_cancelled"] != "t"].drop(columns=["api_id", "season_id", "wikipedia", "is_cancelled"])
round = round.rename(columns={"id": "round_id"})
round = round.rename(columns={col: f"round_{col}" for col in round.columns if "id" not in col})
round = round[["round_name", "round_number", "round_race_number", "round_date", "circuit_id", "round_id"]]
round = round.sort_values(by="round_date", ascending=True)

circuit = endid(circuit)
round = endid(round)

# Export cleaned circuits and rounds
circuit.to_csv("cleaned/Circuit_cleaned.csv", index=False, encoding="utf-8-sig")
round.to_csv("cleaned/Round_cleaned.csv", index=False, encoding="utf-8-sig")

# =========================
# SESSION ENTRIES
# =========================

session_entry = pd.read_csv("formula_one_sessionentry.csv")

cols = ["id", "detail", "grid", "is_classified", "is_eligible_for_points", "laps_completed", "points", "position", "session_id", "time"]

# Keep only entries with a valid grid position
session_entry = session_entry[cols].dropna(subset=["grid"], how="all")

# Convert times to timedeltas
td = pd.to_timedelta(session_entry["time"], errors="coerce")

# Calculate average lap time for each session
avg_lap_by_session = td.div(session_entry["laps_completed"]).groupby(session_entry["session_id"]).transform("mean")

# Calculate total session laps and missing laps for each entry
max_laps = session_entry.groupby("session_id")["laps_completed"].transform("max")
missing_laps = max_laps - session_entry["laps_completed"]

# Estimate missing times using the average known session time
base_session_time = td.groupby(session_entry["session_id"]).transform("mean")
estimated_td = base_session_time + (avg_lap_by_session * missing_laps)
filled_td = td.fillna(estimated_td)

# Format timedeltas as HH:MM:SS.s
def fmt_time(val):
    if pd.isna(val):
        return pd.NA
    total_seconds = val.total_seconds()
    hours = int(total_seconds // 3600)
    minutes = int((total_seconds % 3600) // 60)
    seconds = total_seconds % 60
    return f"{hours:02d}:{minutes:02d}:{seconds:04.1f}"

session_entry["time"] = filled_td.apply(fmt_time)

# Rename columns
session_entry = session_entry.rename(columns={"id": "session_entry_id"})
session_entry = session_entry.rename(columns={col: f"session_entry_{col}" for col in session_entry.columns if "_id" not in col})

# Reorder columns
session_entry = session_entry[["session_entry_time", "session_entry_position", "session_entry_grid", "session_entry_points", "session_entry_laps_completed", "session_entry_detail", "session_entry_is_classified", "session_entry_is_eligible_for_points", "session_entry_id", "session_id"]]
session_entry = endid(session_entry)

# Export cleaned session entries
session_entry.to_csv("cleaned/Session_entry_cleaned.csv", index=False, encoding="utf-8-sig")

# =========================
# SESSIONS
# =========================

session = pd.read_csv("formula_one_session.csv")

cols = ["scheduled_laps", "type", "number", "round_id", "id"]

mask = (session["is_cancelled"] != "t") & (session["scheduled_laps"].notna()) & (session["scheduled_laps"] != "NaN")
session = session.loc[mask, cols]
session = session.rename(columns={"id": "session_id"})
session = session.rename(columns={col: f"session_{col}" for col in session.columns if "id" not in col})

# =========================
# DRIVER CHAMPIONSHIPS
# =========================

driver_championship = pd.read_csv("formula_one_driverchampionship.csv")

cols = ["id", "driver_id", "highest_finish", "points", "position", "session_id", "win_count", "year"]

mask = (driver_championship["is_eligible"] == "t") & (driver_championship["highest_finish"].notna()) & (driver_championship["highest_finish"] != "NaN")
driver_championship = driver_championship.loc[mask, cols]

# Rename columns
driver_championship = driver_championship.rename(columns={"id": "driver_championship_id"})
driver_championship = driver_championship.rename(columns={col: f"driver_championship_{col}" for col in driver_championship.columns if "id" not in col})

# =========================
# MERGE DRIVER CHAMPIONSHIPS
# =========================

# Merge sessions, rounds and circuits to retrieve all related information
driver_championship = pd.merge(driver_championship, session, on="session_id", how="left")
driver_championship = pd.merge(driver_championship, round, on="round_id", how="left")
driver_championship = pd.merge(driver_championship, circuit, on="circuit_id", how="left")

# Remove rows with missing circuit IDs or driver championship positions
driver_championship = driver_championship.dropna(subset=["circuit_id", "driver_championship_position"])
driver_championship = endid(driver_championship)

# Sort rows chronologically by round date
driver_championship = driver_championship.sort_values(by="round_date", ascending=True)

# Export cleaned driver championships
driver_championship.to_csv("cleaned/Driver_Championship_cleaned.csv", index=False, encoding="utf-8-sig")

# =========================
# MERGE SESSIONS AND SESSION ENTRIES
# =========================

# Export cleaned sessions before merging
session.to_csv("cleaned/Session_cleaned.csv", index=False, encoding="utf-8-sig")

session = pd.merge(session, session_entry, left_on="session_id", right_on="session_id", how="left")
session = pd.merge(session, round, left_on="round_id", right_on="round_id", how="left")

# Remove rows without a session entry
session = session.dropna(subset=["session_entry_id"])
session = endid(session)

# Export merged session information
session.to_csv("cleaned/Merged_Session_info.csv", index=False, encoding="utf-8-sig")

# =========================
# MERGE DRIVER CHAMPIONSHIPS AND SESSION ENTRIES
# =========================

# Select relevant session entry columns and remove duplicate session IDs
session_entry_clean = session_entry[["session_id", "session_entry_id", "session_entry_time", "session_entry_position", "session_entry_grid", "session_entry_points", "session_entry_laps_completed", "session_entry_detail", "session_entry_is_classified", "session_entry_is_eligible_for_points"]].dropna(subset=["session_id"]).drop_duplicates(subset=["session_id"], keep="first")

driver_championship = pd.merge(driver_championship, session_entry_clean, on="session_id", how="left")

# =========================
# MERGE DRIVER INFORMATION
# =========================

driver_championship = pd.merge(driver_championship, drivers, left_on="driver_id", right_on="driver_id", how="left")
driver_championship = endid(driver_championship)

# =========================
# FINAL COLUMN SELECTION
# =========================
driver_championship = driver_championship[
    [
        # Driver
        "drivers_forename", "drivers_surname", "drivers_date_of_birth", "drivers_nationality", "drivers_country_code",

        # Championship standings
        "driver_championship_year", "driver_championship_position", "driver_championship_points",
        "driver_championship_win_count", "driver_championship_highest_finish",

        # Round & Circuit
        "round_name", "round_number", "round_race_number", "round_date",
        "circuit_name", "circuit_locality", "circuit_altitude", "circuit_longitude",
        "circuit_country", "circuit_country_code",

        # Session & Session Entry
        "session_type", "session_number", "session_scheduled_laps",
        "session_entry_grid", "session_entry_position", "session_entry_time",
        "session_entry_points", "session_entry_laps_completed", "session_entry_detail",
        "session_entry_is_classified", "session_entry_is_eligible_for_points",

        # IDs
        "driver_championship_id", "driver_id", "round_id", "circuit_id", "session_id", "session_entry_id"
    ]
]

# =========================
# EXPORT FINAL DATASET
# =========================

driver_championship.to_csv("cleaned/Merged_Driver_Championship_info.csv", index=False, encoding="utf-8-sig")

# =========================
# ADDITIONAL CHECKS
# =========================

# session.head(10)

# Check for missing values in the driver championship dataset
# checkforNAN(driver_championship)

# Print driver column names
# print(drivers.columns.tolist())

In [52]:

import pandas as pd

# =========================
# TEAM CHAMPIONSHIPS
# =========================

team_championship = pd.read_csv("formula_one_teamchampionship.csv")

# Remove rows where teams are not eligible or their position is missing
team_championship = team_championship[(team_championship["is_eligible"] != "f") & (team_championship["position"].notnull())]

# Remove unnecessary columns
team_championship = team_championship.drop(columns=["year", "season_id", "highest_finish", "is_eligible", "round_id", "adjustment_type", "session_number", "round_number"])

# Rename columns
team_championship = team_championship.rename(columns={"id": "team_championship_id"})
team_championship = team_championship.rename(columns={col: f"team_championship_{col}" for col in team_championship.columns if "id" not in col})

team_championship = endid(team_championship)


# =========================
# DRIVER TEAM
# =========================

driver_team = pd.read_csv("formula_one_teamdriver.csv").drop(columns=["api_id", "role", "id", "season_id"])

# Create a comma-separated string containing all team members
# Retrieve driver first and last names and combine them into a single string

for team_id in driver_team["team_id"].unique():
    members = drivers[drivers["driver_id"].isin(driver_team[driver_team["team_id"] == team_id]["driver_id"])]
    members_str = ", ".join(members["drivers_forename"] + "_" + members["drivers_surname"] + "_" + members["driver_id"].astype(str))
    driver_team.loc[driver_team["team_id"] == team_id, "team_members_N_S_ID"] = members_str

# Keep only unique team_id rows
driver_team = driver_team.drop_duplicates(subset=["team_id"]).drop(columns=["driver_id"])


# =========================
# MERGE TEAM CHAMPIONSHIPS WITH SESSIONS
# =========================

session = pd.read_csv("cleaned/Session_cleaned.csv")

team_championship = pd.merge(team_championship, session, left_on="session_id", right_on="session_id", how="left")

# Remove rows without a valid session type
team_championship = team_championship.dropna(subset=["session_type"])


# =========================
# MERGE TEAM INFORMATION
# =========================

team_championship = pd.merge(team_championship, teams, left_on="team_id", right_on="team_id", how="left")


# =========================
# MERGE ROUND AND CIRCUIT INFORMATION
# =========================

team_championship = pd.merge(team_championship, round, left_on="round_id", right_on="round_id", how="left")
team_championship = pd.merge(team_championship, circuit, left_on="circuit_id", right_on="circuit_id", how="left")


# =========================
# MERGE DRIVER TEAM INFORMATION
# =========================

team_championship = pd.merge(team_championship, driver_team, left_on="team_id", right_on="team_id", how="left")


# =========================
# FINAL CLEANING AND COLUMN ORDER
# =========================

team_championship = team_championship[
    [
        # Team
        "teams_name", "teams_nationality", "teams_country_code", "team_members_N_S_ID",

        # Points
        "team_championship_points", "team_championship_position", "team_championship_win_count",

        # Circuit
        "circuit_name", "circuit_locality", "circuit_altitude", "circuit_longitude", "circuit_country", "circuit_country_code",

        # Round
        "round_name", "round_number", "round_race_number", "round_date",

        # Session
        "session_scheduled_laps", "session_type", "session_number",

        # IDs
        "team_championship_id", "team_id", "circuit_id", "round_id", "session_id"
    ]
]

# Sort rows chronologically by round date
team_championship = team_championship.sort_values(by="round_date", ascending=True)

# Export final dataset
team_championship.to_csv("cleaned/Merged_Team_Championship_info.csv", index=False, encoding="utf-8-sig")
